# YOLO26 Training - Google Colab

This notebook fine-tunes **Ultralytics YOLO26** (not YOLOv8). Requires **ultralytics >= 8.4.0** so `yolo26*.pt` weights resolve correctly.

Run in Google Colab with GPU enabled: `Runtime > Change runtime type > T4 GPU` (or better).

Upload your dataset zip (for example `YOLO26.zip`) to either:
- `/content` using the Files panel, or
- `Google Drive/MyDrive`

**Pipeline:** run cells **top to bottom** (`Runtime → Run all`). After training, use `runs/.../weights/best.pt` locally with `infer.py` or `app.py` (see repo root).

The last code cell zips training runs and test predictions, then triggers a **browser download** to your local machine. If the download is blocked, copy `yolo26_trained_output.zip` from `/content` via the Files panel, or use the optional Google Drive copy path printed there.

Your `data.yaml` paths must match folder names on disk (`valid` or `val` for the validation split).

In [17]:
%pip install -q 'ultralytics>=8.4.0'

import importlib.metadata as _metadata
from ultralytics import YOLO
import torch

try:
    _uv = _metadata.version('ultralytics')
except Exception:
    _uv = 'unknown'
print('ultralytics version:', _uv)

if not torch.cuda.is_available():
    raise RuntimeError(
        'GPU is not enabled. In Colab, select Runtime > Change runtime type > T4 GPU, then run again.'
    )

DEVICE = 0
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0))
print('Next cell uses pretrained:', 'yolo26s.pt', '(downloaded on first train if missing)')

ultralytics version: 8.4.48
CUDA available: True
GPU: Tesla T4
Next cell uses pretrained: yolo26s.pt (downloaded on first train if missing)


In [18]:
from pathlib import Path
import shutil
import zipfile

try:
    from google.colab import drive
except Exception:
    drive = None

EXTRACT_DIR = Path('/content/yolo26_dataset')
DATASET_ZIP_NAME = 'YOLO26.zip'

# Primary expected location in Colab files.
zip_path = Path('/content') / DATASET_ZIP_NAME
if not zip_path.exists():
    zip_path = None

# Fallback to Google Drive if needed.
if zip_path is None and drive is not None:
    if not Path('/content/drive/MyDrive').exists():
        try:
            drive.mount('/content/drive', force_remount=False)
        except Exception as exc:
            print('Note (Drive mount):', exc)
    drive_zip = Path('/content/drive/MyDrive') / DATASET_ZIP_NAME
    if drive_zip.exists():
        zip_path = drive_zip

# Last fallback: any zip in /content or MyDrive.
if zip_path is None:
    content_any_zip = sorted(Path('/content').glob('*.zip'))
    zip_path = content_any_zip[0] if content_any_zip else None

if zip_path is None and drive is not None and Path('/content/drive/MyDrive').exists():
    drive_any_zip = sorted(Path('/content/drive/MyDrive').glob('*.zip'))
    zip_path = drive_any_zip[0] if drive_any_zip else None

if zip_path is None:
    raise FileNotFoundError(
        'Dataset zip not found. Upload YOLO26.zip to /content or /content/drive/MyDrive and run again.'
    )

print('Using dataset zip:', zip_path)

if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)

print('Extracted dataset to:', EXTRACT_DIR)

Using dataset zip: /content/drive/MyDrive/YOLO26.zip
Extracted dataset to: /content/yolo26_dataset


In [19]:
from pathlib import Path

candidate_yaml_files = list(EXTRACT_DIR.rglob('data.yaml'))
if not candidate_yaml_files:
    raise FileNotFoundError('data.yaml was not found after unzipping.')

# If the zip contains multiple data.yaml files, prefer the one next to train/valid.
def _yaml_score(p: Path) -> tuple:
    d = p.parent
    has_train = (d / 'train' / 'images').is_dir()
    has_val_split = (d / 'valid' / 'images').is_dir() or (d / 'val' / 'images').is_dir()
    return (0 if has_train and has_val_split else 1, len(p.parts), str(p))

DATA_YAML_PATH = sorted(candidate_yaml_files, key=_yaml_score)[0]
if len(candidate_yaml_files) > 1:
    print('Multiple data.yaml found; using:', DATA_YAML_PATH)
DATASET_DIR = DATA_YAML_PATH.parent
DATA_YAML = str(DATA_YAML_PATH)

# Roboflow / YOLO layout: prefer valid/, accept val/ for the validation split folders.
_val_root = DATASET_DIR / 'valid'
if not (_val_root / 'images').is_dir():
    _alt = DATASET_DIR / 'val'
    if (_alt / 'images').is_dir():
        _val_root = _alt
        print('Using val/ instead of valid/ for validation split folders.')

required_dirs = [
    DATASET_DIR / 'train' / 'images',
    DATASET_DIR / 'train' / 'labels',
    _val_root / 'images',
    _val_root / 'labels',
    DATASET_DIR / 'test' / 'images',
    DATASET_DIR / 'test' / 'labels',
]
missing_dirs = [str(path) for path in required_dirs if not path.exists()]
if missing_dirs:
    raise FileNotFoundError('Missing dataset folders: ' + ', '.join(missing_dirs))

print('DATASET_DIR:', DATASET_DIR)
print('DATA_YAML:', DATA_YAML)

DATASET_DIR: /content/yolo26_dataset
DATA_YAML: /content/yolo26_dataset/data.yaml


In [20]:
from pathlib import Path

RUN_NAME = 'yolo26s_512_100epochs'
RUN_PROJECT = '/content/yolo26_runs'

# YOLO26 detection weights: yolo26n/s/m/l/x.pt (see Ultralytics YOLO26 docs)
PRETRAINED = 'yolo26s.pt'

print('Training with:', DATA_YAML)
print('Saving runs to:', f'{RUN_PROJECT}/{RUN_NAME}')
print('Base weights:', PRETRAINED)

model = YOLO(PRETRAINED)

results = model.train(
    task='detect',
    data=DATA_YAML,
    epochs=100,
    imgsz=512,
    batch=8,
    patience=30,
    device=DEVICE,
    project=RUN_PROJECT,
    name=RUN_NAME,
    exist_ok=True,
)

BEST_MODEL_PATH = str(Path(RUN_PROJECT) / RUN_NAME / 'weights' / 'best.pt')
print('Best model saved at:', BEST_MODEL_PATH)

Training with: /content/yolo26_dataset/data.yaml
Saving runs to: /content/yolo26_runs/yolo26s_512_100epochs
Base weights: yolo26s.pt
Ultralytics 8.4.48 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolo26_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26s.pt, momentum=0.937, mos

In [30]:
import csv
import json
import shutil
from pathlib import Path

import numpy as np
from ultralytics import YOLO

try:
    from IPython.display import Image as IPyImage, Markdown, display
except ImportError:
    IPyImage = Markdown = display = None


def _f1(precision, recall):
    if precision is None or recall is None:
        return None
    precision = float(precision)
    recall = float(recall)
    if precision + recall <= 0:
        return 0.0
    return 2.0 * precision * recall / (precision + recall)


def _mean_scalar_or_array(value):
    if value is None:
        return None
    try:
        if isinstance(value, (list, tuple, np.ndarray)):
            array = np.asarray(value, dtype=float)
            array = array[~np.isnan(array)]
            if array.size == 0:
                return None
            return float(np.mean(array))
        return float(value)
    except Exception:
        return None


def _first_non_none(*values):
    for value in values:
        if value is not None:
            return value
    return None


def _get_attr_or_key(obj, key):
    if obj is None:
        return None
    if isinstance(obj, dict):
        return obj.get(key)
    return getattr(obj, key, None)


def _extract_pr_candidates(metrics):
    box = getattr(metrics, 'box', None)
    if box is not None:
        mp = _get_attr_or_key(box, 'mp')
        mr = _get_attr_or_key(box, 'mr')
        if mp is not None or mr is not None:
            return _mean_scalar_or_array(mp), _mean_scalar_or_array(mr), 'box.mp/mr'

        pr = _get_attr_or_key(box, 'pr')
        if isinstance(pr, dict):
            precision = _first_non_none(pr.get('precision'), pr.get('prec'))
            recall = _first_non_none(pr.get('recall'), pr.get('rec'))
            if precision is not None or recall is not None:
                return _mean_scalar_or_array(precision), _mean_scalar_or_array(recall), 'box.pr'

    results_dict = getattr(metrics, 'results_dict', None) or {}
    if isinstance(results_dict, dict):
        precision = _first_non_none(results_dict.get('precision'), results_dict.get('P'), results_dict.get('prec'))
        recall = _first_non_none(results_dict.get('recall'), results_dict.get('R'))
        if precision is not None or recall is not None:
            return _mean_scalar_or_array(precision), _mean_scalar_or_array(recall), 'results_dict'

    precision = _first_non_none(_get_attr_or_key(metrics, 'precision'), _get_attr_or_key(metrics, 'prec'))
    recall = _get_attr_or_key(metrics, 'recall')
    if precision is not None or recall is not None:
        return _mean_scalar_or_array(precision), _mean_scalar_or_array(recall), 'metrics.attr'

    return None, None, None


if not Path(BEST_MODEL_PATH).exists():
    raise FileNotFoundError(f'Best model not found: {BEST_MODEL_PATH}. Run training first.')

best_model = YOLO(BEST_MODEL_PATH)

# Evaluation artifacts directory
_METRICS_PROJECT = Path(RUN_PROJECT) / RUN_NAME / 'final_eval'
_METRICS_PROJECT.mkdir(parents=True, exist_ok=True)

# Run validation on validation and test sets
val_metrics = best_model.val(
    data=DATA_YAML,
    split='val',
    imgsz=512,
    device=DEVICE,
    task='detect',
    plots=True,
    save_json=True,
    verbose=True,
    project=str(_METRICS_PROJECT),
    name='val_eval',
    exist_ok=True,
)

test_metrics = best_model.val(
    data=DATA_YAML,
    split='test',
    imgsz=512,
    device=DEVICE,
    task='detect',
    plots=True,
    save_json=True,
    verbose=True,
    project=str(_METRICS_PROJECT),
    name='test_eval',
    exist_ok=True,
)


def _print_res(prefix, metrics):
    precision, recall, source = _extract_pr_candidates(metrics)

    if (precision is None or recall is None) and getattr(metrics, 'results_dict', None):
        results_dict = getattr(metrics, 'results_dict') or {}
        if precision is None:
            precision = _mean_scalar_or_array(_first_non_none(results_dict.get('precision'), results_dict.get('P'), results_dict.get('prec')))
        if recall is None:
            recall = _mean_scalar_or_array(_first_non_none(results_dict.get('recall'), results_dict.get('R')))
        if source is None and (precision is not None or recall is not None):
            source = 'results_dict'

    box = getattr(metrics, 'box', None)
    map50 = _mean_scalar_or_array(getattr(box, 'map50', None) if box is not None else None)
    map_all = _mean_scalar_or_array(getattr(box, 'map', None) if box is not None else None)

    print(f'\n{prefix}')
    print(f"  Precision : {precision:.4f}" if precision is not None else '  Precision : N/A')
    print(f"  Recall    : {recall:.4f}" if recall is not None else '  Recall    : N/A')
    f1_value = _f1(precision, recall)
    print(f"  F1        : {f1_value:.4f}" if f1_value is not None else '  F1        : N/A')
    print(f"  mAP50     : {map50:.4f}" if map50 is not None else '  mAP50     : N/A')
    print(f"  mAP50-95  : {map_all:.4f}" if map_all is not None else '  mAP50-95  : N/A')
    if source:
        print(f'  (source: {source})')


_print_res('Validation metrics', val_metrics)
_print_res('Test metrics', test_metrics)

print('\nMetric files saved under:', _METRICS_PROJECT)

Ultralytics 8.4.48 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO26s summary (fused): 122 layers, 9,467,115 parameters, 0 gradients, 20.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1312.8±189.8 MB/s, size: 36.0 KB)
val: Scanning /content/yolo26_dataset/valid/labels.cache... 3 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 3/3 1.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 12.3it/s 0.1s
                   all          3          3      0.935          1      0.995      0.493
            Vegetative          3          3      0.935          1      0.995      0.493
Speed: 0.8ms preprocess, 11.4ms inference, 0.0ms loss, 0.2ms postprocess per image
Saving /content/yolo26_runs/yolo26s_512_100epochs/final_eval/val_eval/predictions.json...
Results saved to /content/yolo26_runs/yolo26s_512_100epochs/final_eval/val_eval
Ultralytics 8.4.48 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:

In [22]:
from pathlib import Path

PRED_PROJECT = '/content/yolo26_predictions'
PRED_NAME = 'test_predictions'

test_images_dir = str(Path(DATASET_DIR) / 'test' / 'images')
if not Path(test_images_dir).exists():
    raise FileNotFoundError(f'Test images directory not found: {test_images_dir}')

pred_results = best_model.predict(
    source=test_images_dir,
    imgsz=512,
    conf=0.25,
    device=DEVICE,
    save=True,
    project=PRED_PROJECT,
    name=PRED_NAME,
    exist_ok=True,
)

PRED_DIR = f'{PRED_PROJECT}/{PRED_NAME}'
print('Predictions saved at:', PRED_DIR)


image 1/12 /content/yolo26_dataset/test/images/1-2-_jpeg.rf.6934e3d55dd1da958de4b7b212cf6a65.jpg: 512x512 2 Vegetatives, 12.1ms
image 2/12 /content/yolo26_dataset/test/images/20260502_161648_jpg.rf.22bb6166c3bbb7f48eaaac970b839b05.jpg: 512x512 2 Vegetatives, 14.7ms
image 3/12 /content/yolo26_dataset/test/images/27638_jpeg.rf.186562bd638d326d059543ffde5ae522.jpg: 512x512 1 Flowering, 1 Vegetative, 12.1ms
image 4/12 /content/yolo26_dataset/test/images/2_jpeg.rf.fe7ec1dc9cddfb1a918c26f47a9a2064.jpg: 512x512 (no detections), 12.1ms
image 5/12 /content/yolo26_dataset/test/images/3_jpg.rf.2575565aa2b3ccb92aa50e3c50348fa8.jpg: 512x512 2 Mature_Canes, 12.1ms
image 6/12 /content/yolo26_dataset/test/images/5-2-_jpg.rf.b8cab6b81856f0ae8e3cbefeb5404f4a.jpg: 512x512 1 Vegetative, 12.0ms
image 7/12 /content/yolo26_dataset/test/images/5_jpg.rf.d4a6a295bf3b8ace57df33cde71a6080.jpg: 512x512 (no detections), 12.0ms
image 8/12 /content/yolo26_dataset/test/images/6_jpg.rf.a11fbb082fe5ead4abc71915d01767a5

In [23]:
from pathlib import Path
import shutil
import zipfile

try:
    from google.colab import files as colab_files
except Exception:
    colab_files = None

run_dir = Path(RUN_PROJECT) / RUN_NAME
pred_dir = Path(PRED_DIR)
metrics_dir = Path(RUN_PROJECT) / RUN_NAME / 'final_eval'
export_zip = Path('/content/yolo26_trained_output.zip')

if export_zip.exists():
    export_zip.unlink()

best_on_disk = run_dir / 'weights' / 'best.pt'
if not best_on_disk.is_file():
    print('WARNING: best.pt not found at', best_on_disk, '(train or RUN_NAME may not match).')
else:
    print(
        'best.pt on Colab disk:',
        best_on_disk,
        '|',
        round(best_on_disk.stat().st_size / (1024 * 1024), 2),
        'MB',
    )

with zipfile.ZipFile(export_zip, 'w', compression=zipfile.ZIP_DEFLATED) as zip_file:
    for prefix, folder in [('runs', run_dir), ('predictions', pred_dir), ('metrics_eval', metrics_dir)]:
        if not folder.exists():
            print('Skip (missing):', folder)
            continue
        for file_path in folder.rglob('*'):
            if file_path.is_file():
                arcname = Path(prefix) / file_path.relative_to(folder)
                zip_file.write(file_path, arcname=arcname.as_posix())

with zipfile.ZipFile(export_zip, 'r') as zf:
    best_in_zip = [n for n in zf.namelist() if n.endswith('weights/best.pt')]
    print('Created:', export_zip)
    print('Zip size MB:', round(export_zip.stat().st_size / (1024 * 1024), 2))
    if best_in_zip:
        info = zf.getinfo(best_in_zip[0])
        print(
            'best.pt inside zip:',
            best_in_zip[0],
            '|',
            round(info.file_size / (1024 * 1024), 2),
            'MB uncompressed',
        )
    else:
        print('WARNING: no path ending with weights/best.pt inside the zip; check RUN_PROJECT / RUN_NAME.')

# 1) Browser download to your local device (Colab)
if colab_files is not None:
    try:
        colab_files.download(str(export_zip))
        print('Download started in your browser.')
    except Exception as exc:
        print('Automatic download failed:', exc)
        print('Use Files panel: right-click', export_zip, '-> Download')
else:
    print('Not running in Colab; zip is at:', export_zip)

# 2) Optional backup to Google Drive (persists if Colab disconnects)
drive_zip = Path('/content/drive/MyDrive/yolo26_trained_output.zip')
if Path('/content/drive/MyDrive').exists():
    try:
        shutil.copy2(export_zip, drive_zip)
        print('Also copied to Drive:', drive_zip)
    except Exception as exc:
        print('Drive copy skipped:', exc)

best.pt on Colab disk: /content/yolo26_runs/yolo26s_512_100epochs/weights/best.pt | 19.36 MB
Created: /content/yolo26_trained_output.zip
Zip size MB: 42.93
best.pt inside zip: runs/weights/best.pt | 19.36 MB uncompressed


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download started in your browser.
Also copied to Drive: /content/drive/MyDrive/yolo26_trained_output.zip
